# ITO5202 Data processing for Big data

##  Activity: Parallel Joins in Spark DataFrames

For this tutorial we will look at the high level join strategies used in Spark while performing join operation. We will try to understand the interal working of these strategies by examining the query execution plan and the graphical plan provided in Spark UI. We will then look into various other basic join algorithms used by Spark.

Let's get started.


## Table of Contents

* [SparkContext and SparkSession](#one)
* [Parallel Join Strategies](#parallel-join)
    * [Broadcast Hash Join](#bhj)
    * [Sort Merge Join](#smj)
* [Parallel Joins](#other-joins)
    * [Inner Join](#inner)        
    * [Left Join](#left)        
    * [Full Outer Join](#full_outer)        
    * [Left Semi Join](#left_semi)        
    * [Left Anti Join](#left_anti)            
* [Lab Tasks](#lab-task-1)
    * [Lab Task 1](#lab-task-1)
    * [Lab Task 2](#lab-task-2)
    * [Lab Task 3](#lab-task-3)    

<a class="anchor" id="one"></a>
## Import Spark classes and create Spark Context

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#006DAE">TODO: </strong>In the cell block below, initialize the spark session named as <strong>spark</strong> and create the <code>SparkContext</code> names <strong>sc</strong> from that Spark Session.
    
<p><strong style="color:red">Important:</strong> You cannot proceed to other steps without completing this.</p>
</div>

In [2]:
# Import SparkConf class into program
from pyspark import SparkConf

# local[*]: run Spark in local mode with as many working processors as logical cores on your machine
# If we want Spark to run locally with 'k' worker threads, we can specify as "local[k]".
master = "local[*]"
# The `appName` field is a name to be shown on the Spark cluster UI page
app_name = "Parallel Join"
# Setup configuration parameters for Spark
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

# Import SparkSession classes 
from pyspark.sql import SparkSession # Spark SQL

#TODO : Initialize Spark Session and create a SparkContext Object
# Initialize Spark Session
spark = SparkSession.builder \
    .config(conf=spark_conf) \
    .getOrCreate()

# Create SparkContext Object
sc = spark.sparkContext


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/29 19:37:25 WARN Utils: Your hostname, Stefans-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.121 instead (on interface en0)
26/08/29 19:37:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/envs/ITO5202/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/29 19:37:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/29 19:37:26 WARN Utils: Service '

<a class="anchor" id="parallel-join"></a>
## Parallel Join Strategies
### Creating  datasets (i.e. dataframes)
In the code below, we are creating two datasets with a common key "id". When we are joining two tables, we need at least 1 common key.

In [3]:
###### Setting dataset
import random
random.seed(0)

# List of tuples
tableA = [(i,'A'+str(i)) for i in range(100,110)]
tableB = [(i,'B'+str(i)) for i in range(10,1000)]
# Shuffle the lists to not have it ordered
random.shuffle(tableA)
random.shuffle(tableB)

# Converting to dataframe each list of tuples
df_A = spark.createDataFrame(tableA , ["id", "valueA"])
df_A.show()
df_B = spark.createDataFrame(tableB , ["id", "valueB"])
df_B.show()

+---+------+
| id|valueA|
+---+------+
|107|  A107|
|108|  A108|
|101|  A101|
|105|  A105|
|103|  A103|
|104|  A104|
|102|  A102|
|100|  A100|
|109|  A109|
|106|  A106|
+---+------+

+---+------+
| id|valueB|
+---+------+
|450|  B450|
|633|  B633|
|170|  B170|
|561|  B561|
|247|  B247|
|400|  B400|
|590|  B590|
|290|  B290|
|134|  B134|
|739|  B739|
|991|  B991|
|504|  B504|
|241|  B241|
|311|  B311|
|964|  B964|
|669|  B669|
|928|  B928|
|831|  B831|
|270|  B270|
|119|  B119|
+---+------+
only showing top 20 rows


<a class="anchor" id="bhj"></a>
### 1. Broadcast Hash Join
In this type of join, one dataset(the smaller one) is broadcasted (sent over) to each executor. By doing this, we can avoid the shuffle for the other larger dataset. Not doing the shuffle increase the speed of the join operation.

<i>We need to use the broadcast function inside the join to broadcast the table</i>

In [4]:
from pyspark.sql.functions import broadcast

# Use broadcast function to specify the use of BroadcastHashJoin algorithm
df_joined_broadcast = df_B.join(df_A,df_A.id==df_B.id,how='inner')
df_joined_broadcast.show()

+---+------+---+------+
| id|valueB| id|valueA|
+---+------+---+------+
|107|  B107|107|  A107|
|108|  B108|108|  A108|
|101|  B101|101|  A101|
|105|  B105|105|  A105|
|103|  B103|103|  A103|
|104|  B104|104|  A104|
|102|  B102|102|  A102|
|100|  B100|100|  A100|
|109|  B109|109|  A109|
|106|  B106|106|  A106|
+---+------+---+------+



#### Query execution plan

In [5]:
## Show execution plan using function explain()
df_joined_broadcast.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [id#9L], [id#0L], Inner
   :- Sort [id#9L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(id#9L, 200), ENSURE_REQUIREMENTS, [plan_id=177]
   :     +- Filter isnotnull(id#9L)
   :        +- Scan ExistingRDD[id#9L,valueB#10]
   +- Sort [id#0L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(id#0L, 200), ENSURE_REQUIREMENTS, [plan_id=178]
         +- Filter isnotnull(id#0L)
            +- Scan ExistingRDD[id#0L,valueA#1]




#### Explanation Query Plan with Broadcast Hash Join
The order of execution goes from top to bottom. The steps are:
1. Scan dataframe A (left side)
  - Filter id not null in dataframe A
2. Scan dataframe B (right side)
  - Filter id not null in dataframe B
3. Broadcast dataframe B: Send dataframe B to each each partition
4. BroadcastHashJoin: Perform join between each partition and the broadcasted dataframe B
5. Project: Select the attributes from both dataframes (df_A: id,valueA and df_b: id,valueB)
6. Collect all the results to the driver

#### Graphical Execution plan in Spark UI
Go to your <strong>Spark UI</strong> and Click on the <strong>SQL</strong> tab to view the graphical equivalent of the above physical plan.
<img src="attachment:image.png" width="70%">

<a class="anchor" id="smj"></a>
### 2. Sort Merge Join
In this join approach, the datasets are sorted first and the second operation merges the sorted data in the partition. This is the <strong>default</strong> join algorithm used by spark.

In [6]:
df_joined_sortmerge = df_A.join(df_B,df_A.id==df_B.id,how='inner')
df_joined_sortmerge.show()

+---+------+---+------+
| id|valueA| id|valueB|
+---+------+---+------+
|107|  A107|107|  B107|
|108|  A108|108|  B108|
|101|  A101|101|  B101|
|105|  A105|105|  B105|
|103|  A103|103|  B103|
|104|  A104|104|  B104|
|102|  A102|102|  B102|
|100|  A100|100|  B100|
|109|  A109|109|  B109|
|106|  A106|106|  B106|
+---+------+---+------+



#### Physical Execution Plan

In [7]:
df_joined_sortmerge.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [id#0L], [id#9L], Inner
   :- Sort [id#0L ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(id#0L, 200), ENSURE_REQUIREMENTS, [plan_id=335]
   :     +- Filter isnotnull(id#0L)
   :        +- Scan ExistingRDD[id#0L,valueA#1]
   +- Sort [id#9L ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(id#9L, 200), ENSURE_REQUIREMENTS, [plan_id=336]
         +- Filter isnotnull(id#9L)
            +- Scan ExistingRDD[id#9L,valueB#10]




#### Graphical Execution Plan in Spark UI
Go to your <strong>Spark UI</strong> and Click on the <strong>SQL</strong> tab to view the graphical equivalent of the above physical plan.
![image.png](attachment:image.png)

#### Explanation Query Plan with Sort Merge Join
The order of execution goes from top to bottom. The steps are:
1. Scan dataframe A (left side)
  - Filter id not null in dataframe A
2. Scan dataframe B (right side)
  - Filter id not null in dataframe B
3. Exchange dataframe A: Partition dataframe A with hash partitioning
4. Exchange dataframe B: Partition dataframe B with hash partitioning
5. Sort dataframe A: Sort data within each partition
6. Sort dataframe B: Sort data within each partition
7. Perform Sort Merge Join between both dataframes
5. Project: Select the attributes from both dataframes (df_A: id,valueA and df_b: id,valueB)
6. Collect all the results to the driver

<a class="anchor" id="other-joins"></a>
## Parallel Join

Now we will implement multiple join operations and visualise the parallelism embedded in Spark to perform these kind of queries. The join queries that we will perform are:
1. Inner Join
1. Left Join
1. Full Outer Join
1. Left Semi Join
1. Left Anti Join

**All left operations have their right operations as well, but this is a commutative operation so we will focus only on left operations** 

In this tutorial, you will use three csv files as datasets which contains the information of the Summer Olympics (summer.csv) and Winter Olympics (winter.csv) plus the information of the list of countries (dictionary.csv).

In [8]:
# Read files into dataframes
df_dictionary = spark.read.csv("dictionary.csv",header=True)
df_summer = spark.read.csv("summer.csv",header=True).repartition(4)
df_winter = spark.read.csv("winter.csv",header=True).repartition(4)

# Create Views from Dataframes
df_dictionary.createOrReplaceTempView("sql_dictionary")
df_summer.createOrReplaceTempView("sql_summer")
df_winter.createOrReplaceTempView("sql_winter")

<a class="anchor" id="lab-task-1"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">1. Lab Task: </strong>In the following code block, display the number of partitions and the schema of the above three dataframes. <strong style="color:#FF5555">Examine how the method <code>repartition</code> is used here. What happens if we do not use repartition?</strong></div>


In [9]:
print(f"####### DICTIONARY INFO:")
print("Number of partitions:", df_dictionary.rdd.getNumPartitions())
df_dictionary.printSchema()

print(f"####### SUMMER INFO:")
print("Number of partitions:", df_summer.rdd.getNumPartitions())
df_summer.printSchema()

print(f"####### WINTER INFO:")
print("Number of partitions:", df_winter.rdd.getNumPartitions())
df_winter.printSchema()

####### DICTIONARY INFO:
Number of partitions: 1
root
 |-- Country: string (nullable = true)
 |-- Code: string (nullable = true)
 |-- Population: string (nullable = true)
 |-- GDP per Capita: string (nullable = true)

####### SUMMER INFO:
Number of partitions: 4
root
 |-- Year: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Sport: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Athlete: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Event: string (nullable = true)
 |-- Medal: string (nullable = true)

####### WINTER INFO:
Number of partitions: 4
root
 |-- Year: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Sport: string (nullable = true)
 |-- Discipline: string (nullable = true)
 |-- Athlete: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Event: string (nullable = true)
 |-- Medal: string (nullable = true)

<a class="anchor" id="inner"></a>
## 1. Inner Join
This join operation returns the result set that have matching values in both dataframes.

In [10]:
#### Join summer and dictionary using Dataframes
df_dict_inner_summ = df_dictionary.join(df_summer,df_dictionary.Code==df_summer.Country,how='inner')
print(df_dict_inner_summ.count())
df_dict_inner_summ.show()

## Join summer and dictionary using SQL
sql_dict_inner_summ = spark.sql('''
  SELECT d.*,w.*
  FROM sql_dictionary d JOIN sql_summer w
  ON d.Code=w.Country
''')
print(sql_dict_inner_summ.count())
sql_dict_inner_summ.show()

25742
+-------------------+----+----------+----------------+----+-----------+-------------+---------------+--------------------+-------+------+--------------------+------+
|            Country|Code|Population|  GDP per Capita|Year|       City|        Sport|     Discipline|             Athlete|Country|Gender|               Event| Medal|
+-------------------+----+----------+----------------+----+-----------+-------------+---------------+--------------------+-------+------+--------------------+------+
|             France| FRA|  66808385|36205.5681017036|2004|     Athens|   Gymnastics|    Artistic G.|    LEPENNEC, Emilie|    FRA| Women|         Uneven Bars|  Gold|
|            Denmark| DEN|   5676002|51989.2934712354|1992|  Barcelona|      Cycling|  Cycling Track| NILSEN, Klaus Kynde|    DEN|   Men|Team Pursuit (4000M)|Bronze|
|      United States| USA| 321418820|56115.7184261955|2008|    Beijing|     Softball|       Softball|      ABBOTT, Monica|    USA| Women|            Softball|Silver

In [11]:
# Now look at the exceution plan for the 2 previous objects
df_dict_inner_summ.explain()
sql_dict_inner_summ.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [Code#62], [Country#87], Inner, BuildLeft, false, false
   :- BroadcastExchange HashedRelationBroadcastMode(List(input[1, string, false]),false), [plan_id=953]
   :  +- Filter isnotnull(Code#62)
   :     +- FileScan csv [Country#61,Code#62,Population#63,GDP per Capita#64] Batched: false, DataFilters: [isnotnull(Code#62)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/stefangarevski/Desktop/Data Processing for Big Data ITO520..., PartitionFilters: [], PushedFilters: [IsNotNull(Code)], ReadSchema: struct<Country:string,Code:string,Population:string,GDP per Capita:string>
   +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=951]
      +- Filter isnotnull(Country#87)
         +- FileScan csv [Year#82,City#83,Sport#84,Discipline#85,Athlete#86,Country#87,Gender#88,Event#89,Medal#90] Batched: false, DataFilters: [isnotnull(Country#87)], Format: CSV, Location: InMemoryFileIndex(1 path

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#006DAE">TODO: </strong>By looking at the Physical Plan, try to understand the internal workings of joins in dataframe. Discuss this with your tutor.</div>

<a class="anchor" id="left"></a>
## 2. Left Join
This join operation returns all records from the left dataframe and the matched records from the right dataframe.

In [12]:
from pyspark.sql.functions import col

#### Join summer and dictionary using Dataframes
df_dict_left_summ = df_dictionary.join(df_summer,df_dictionary.Code==df_summer.Country,how='left')
# df_dict_inner_summ = df_dict_inner_summ.filter(col('Discipline').isNull())
print(df_dict_left_summ.count())
df_dict_left_summ.show()

## Join summer and dictionary using SQL
sql_dict_left_summ = spark.sql('''
  SELECT d.*,w.*
  FROM sql_dictionary d LEFT JOIN sql_summer w
  ON d.Code=w.Country
''')
print(sql_dict_left_summ.count())
sql_dict_left_summ.show()

25814
+---------------+----+----------+----------------+----+-----------+---------+----------+--------------------+-------+------+--------------------+------+
|        Country|Code|Population|  GDP per Capita|Year|       City|    Sport|Discipline|             Athlete|Country|Gender|               Event| Medal|
+---------------+----+----------+----------------+----+-----------+---------+----------+--------------------+-------+------+--------------------+------+
|    Afghanistan| AFG|  32526562|594.323081219966|2012|     London|Taekwondo| Taekwondo|    NIKPAI, Rohullah|    AFG|   Men|          58 - 68 KG|Bronze|
|    Afghanistan| AFG|  32526562|594.323081219966|2008|    Beijing|Taekwondo| Taekwondo|    NIKPAI, Rohullah|    AFG|   Men|             - 58 KG|Bronze|
|        Albania| ALB|   2889167|3945.21758150914|NULL|       NULL|     NULL|      NULL|                NULL|   NULL|  NULL|                NULL|  NULL|
|        Algeria| ALG|  39666519|4206.03123244958|2008|    Beijing|     Judo

In [13]:
# Now look at the exceution plan for the 2 previous objects
df_dict_left_summ.explain()
df_dict_left_summ.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [Code#62], [Country#87], LeftOuter, BuildRight, false, false
   :- FileScan csv [Country#61,Code#62,Population#63,GDP per Capita#64] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/stefangarevski/Desktop/Data Processing for Big Data ITO520..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Country:string,Code:string,Population:string,GDP per Capita:string>
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[5, string, false]),false), [plan_id=1519]
      +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=1517]
         +- Filter isnotnull(Country#87)
            +- FileScan csv [Year#82,City#83,Sport#84,Discipline#85,Athlete#86,Country#87,Gender#88,Event#89,Medal#90] Batched: false, DataFilters: [isnotnull(Country#87)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/stefangarevski/Desktop/Data Processing for 

<a class="anchor" id="full-outer"></a>
## 3. Full Outer Join
This join operation returns a result set that includes rows from both left and right dataframes.

In [14]:
#### Join summer and dictionary using Dataframes
df_dict_outer_summ = df_dictionary.join(df_summer,df_dictionary.Code==df_summer.Country,how='outer')
print(df_dict_outer_summ.count())
df_dict_outer_summ.show()

## Join summer and dictionary using SQL
sql_dict_outer_summ = spark.sql('''
  SELECT d.*,w.*
  FROM sql_dictionary d FULL OUTER JOIN sql_summer w
  ON d.Code=w.Country
''')
print(sql_dict_outer_summ.count())
sql_dict_outer_summ.show()

31237
+--------------------+----+----------+----------------+----+-----------+-------------+-------------------+--------------------+-------+------+--------------------+------+
|             Country|Code|Population|  GDP per Capita|Year|       City|        Sport|         Discipline|             Athlete|Country|Gender|               Event| Medal|
+--------------------+----+----------+----------------+----+-----------+-------------+-------------------+--------------------+-------+------+--------------------+------+
|                NULL|NULL|      NULL|            NULL|2012|     London|    Athletics|          Athletics|             Pending|   NULL| Women|               1500M|  Gold|
|                NULL|NULL|      NULL|            NULL|2012|     London|    Wrestling|Wrestling Freestyle|     KUDUKHOV, Besik|   NULL|   Men|            Wf 60 KG|Silver|
|                NULL|NULL|      NULL|            NULL|2012|     London|Weightlifting|      Weightlifting|             Pending|   NULL|   M

In [15]:
# Now look at the exceution plan for the 2 previous objects
df_dict_outer_summ.explain()
sql_dict_outer_summ.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [Code#62], [Country#87], FullOuter
   :- Sort [Code#62 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(Code#62, 200), ENSURE_REQUIREMENTS, [plan_id=2210]
   :     +- FileScan csv [Country#61,Code#62,Population#63,GDP per Capita#64] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/stefangarevski/Desktop/Data Processing for Big Data ITO520..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Country:string,Code:string,Population:string,GDP per Capita:string>
   +- Sort [Country#87 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(Country#87, 200), ENSURE_REQUIREMENTS, [plan_id=2211]
         +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=2206]
            +- FileScan csv [Year#82,City#83,Sport#84,Discipline#85,Athlete#86,Country#87,Gender#88,Event#89,Medal#90] Batched: false, DataFilters: [], Format: CSV, Location

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#006DAE">TODO: Execution plan comparison</strong>
    Now dive into the execution plan of the previous 3 joins and their parallelism
The objects that will be analysed and compared will be:
    <ul>
        <li>df_dict_inner_summ</li>
<li>df_dict_left_summ</li>
<li>df_dict_outer_summ</li>
    </ul>
The comparisons and analysis can be done using the Spark UI. Compare them after running the next code block. If preferred, you can run them one by one to see in the Jobs </div>



In [16]:
# These actions will execute the Query plan for each of the dataframes
df_dict_inner_summ.collect()
df_dict_left_summ.collect()
df_dict_outer_summ.collect()

## TODO: Look deep into what are the operations performed when an inner join operation is executed.
## For this additional information and better visualisation, go to the Spark UI -> SQL option

[Row(Country=None, Code=None, Population=None, GDP per Capita=None, Year='2012', City='London', Sport='Athletics', Discipline='Athletics', Athlete='Pending', Country=None, Gender='Women', Event='1500M', Medal='Gold'),
 Row(Country=None, Code=None, Population=None, GDP per Capita=None, Year='2012', City='London', Sport='Wrestling', Discipline='Wrestling Freestyle', Athlete='KUDUKHOV, Besik', Country=None, Gender='Men', Event='Wf 60 KG', Medal='Silver'),
 Row(Country=None, Code=None, Population=None, GDP per Capita=None, Year='2012', City='London', Sport='Weightlifting', Discipline='Weightlifting', Athlete='Pending', Country=None, Gender='Men', Event='94KG', Medal='Silver'),
 Row(Country=None, Code=None, Population=None, GDP per Capita=None, Year='2012', City='London', Sport='Weightlifting', Discipline='Weightlifting', Athlete='Pending', Country=None, Gender='Women', Event='63KG', Medal='Gold'),
 Row(Country='Afghanistan', Code='AFG', Population='32526562', GDP per Capita='594.3230812199

<a class="anchor" id="left_semi"></a>
## 4. Left Semi Join
This join operation is like an inner join, but only the left dataframe columns and values are selected

<a class="anchor" id="lab-task-2"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">2. Lab Task: </strong> Implement the <strong>left_semi</strong> join in <strong>Spark SQL.</strong> Ensure that the output from both the approaches is same.</div>


In [17]:
#### Join summer and dictionary using Dataframes
df_dict_semi_summ = df_dictionary.join(df_summer,df_dictionary.Code==df_summer.Country,how='left_semi')
print(df_dict_semi_summ.count())
df_dict_semi_summ.show()

## TODO: Implement the SQL to perform left semi join between summer and dictionary using SQL
df_dict_semi_summ_sql = spark.sql("""
    SELECT d.*
    FROM sql_dictionary d
    LEFT SEMI JOIN sql_summer s
        ON d.Code = s.Country
""")

print(df_dict_semi_summ_sql.count())
df_dict_semi_summ_sql.show()

129
+-----------+----+----------+----------------+
|    Country|Code|Population|  GDP per Capita|
+-----------+----+----------+----------------+
|Afghanistan| AFG|  32526562|594.323081219966|
|    Algeria| ALG|  39666519|4206.03123244958|
|  Argentina| ARG|  43416755|13431.8783398577|
|    Armenia| ARM|   3017712|3489.12768956995|
|  Australia| AUS|  23781169|56310.9629933721|
|    Austria| AUT|   8611088| 43774.985173612|
| Azerbaijan| AZE|   9651349|5496.34464026248|
|    Bahamas| BAH|    388019|22817.2308572518|
|    Bahrain| BRN|   1377237|22600.2140981035|
|   Barbados| BAR|    284215|15429.3404640853|
|    Belarus| BLR|   9513000|5740.45649479562|
|    Belgium| BEL|  11285721|40324.0277657215|
|   Bermuda*| BER|     65235|            NULL|
|   Botswana| BOT|   2262485|6360.13822018837|
|     Brazil| BRA| 207847528| 8538.5899749574|
|   Bulgaria| BUL|   7177991|6993.47735975728|
|    Burundi| BDI|  11178921|277.068309170914|
|   Cameroon| CMR|  23344179|1217.26067048427|
|     Can

<a class="anchor" id="left_anti"></a>
## 5. Left Anti Join
This join operation is the difference of the left dataframe minus the right dataframe, as it selects all rows from df1 that are not present in df2

<a class="anchor" id="lab-task-3"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">3. Lab Task: </strong> Implement the <strong>left_anti</strong> join in <strong>Spark SQL.</strong> Ensure that the output from both the approaches is same.</div>


In [18]:
#### Join summer and dictionary using Dataframes
df_dict_anti_summ = df_dictionary.join(df_summer,df_dictionary.Code==df_summer.Country,how='left_anti')
print(df_dict_anti_summ.count())
df_dict_anti_summ.show()

## TODO: Implement the SQL to perform left anti join between summer and dictionary using SQL
df_dict_anti_summ_sql = spark.sql("""
    SELECT d.*
    FROM sql_dictionary d
    LEFT ANTI JOIN sql_summer s
        ON d.Code = s.Country
""")

print(df_dict_anti_summ_sql.count())
df_dict_anti_summ_sql.show()

72
+--------------------+----+----------+----------------+
|             Country|Code|Population|  GDP per Capita|
+--------------------+----+----------+----------------+
|             Albania| ALB|   2889167|3945.21758150914|
|     American Samoa*| ASA|     55538|            NULL|
|             Andorra| AND|     70473|            NULL|
|              Angola| ANG|  25021974|4101.47215182964|
| Antigua and Barbuda| ANT|     91818|13714.7319616988|
|              Aruba*| ARU|    103889|            NULL|
|          Bangladesh| BAN| 160995642|1211.70153057661|
|              Belize| BIZ|    359287|4878.72126724745|
|               Benin| BEN|  10879829|762.051205441965|
|              Bhutan| BHU|    774830|2655.99889161858|
|             Bolivia| BOL|  10724705|3076.79181060881|
|Bosnia and Herzeg...| BIH|   3810416| 4249.3303131949|
|British Virgin Is...| IVB|     30117|            NULL|
|              Brunei| BRU|    423188|30554.7296658073|
|        Burkina Faso| BUR|  18105570|589.774

**Congratulations on finishing this activity!**

Having practiced today's activities, we're now ready to embark on a trip of the rest of exiciting FIT5202 activities! See you next week!